# FinIA-Flex — Paso 4: Diseño del Prompt Maestro y Técnicas de Prompting

**Proyecto:** FinIA-Flex — Copiloto Financiero y de Control de Costos para manufactura
**Contexto académico:** Caso Práctico Unidad 1, materia Generative IA — Maestría en Ciencia
de Datos y Analítica Visual, Instituto Europeo de Posgrado

---

## Objetivo del notebook

Diseñar y probar el prompt maestro que guía al LLM para generar reportes ejecutivos de
variación presupuestal, cumpliendo el requisito 3 del caso práctico ("implementar técnicas
de prompting para guiar al LLM en la generación de salidas alineadas con las preferencias y
requerimientos").

Este notebook prueba el prompt de forma aislada, **sin conectar aún la recuperación RAG**
(eso corresponde al Paso 5). Aquí se valida que, dado un fragmento de contexto ya recuperado
manualmente, el LLM produce una salida en el formato correcto.

## Técnicas de prompting aplicadas

1. **Role prompting** — se asigna al modelo el rol de analista financiero senior.
2. **Salida estructurada obligatoria** — el prompt exige una estructura fija, alineada al
   documento POL-FIN-003 (Formato Estándar de Reporte Ejecutivo).
3. **Few-shot prompting** — se incluye un ejemplo completo de entrada/salida correctamente
   resuelto, tomado del propio documento de formato.
4. **Chain-of-thought guiado** — se pide al modelo razonar internamente en pasos (clasificar
   la variación, verificar si activa una política, y solo entonces redactar) antes de producir
   la salida final, reduciendo la probabilidad de recomendaciones no sustentadas en los datos.
5. **Grounding explícito** — el prompt instruye al modelo a citar la política aplicable
   únicamente si aparece en el contexto recuperado, y a indicar explícitamente cuando no hay
   una política aplicable, en lugar de inventar una.

## Proveedor de LLM utilizado en las pruebas

Para evitar costo en esta etapa del prototipo, este notebook usa la **API gratuita de Groq**
(modelos Llama en infraestructura de alta velocidad). El prompt es independiente del
proveedor: la misma plantilla funciona con la API de Anthropic (Claude) u OpenAI si se
dispone de acceso, cambiando únicamente la celda de conexión en la Sección 3.


## 1. Instalación y configuración

Se requiere una API key gratuita de Groq (https://console.groq.com/keys). El valor no debe
escribirse directamente en el notebook; se solicita de forma segura con `getpass` para evitar
que quede expuesto en el archivo o en el repositorio de evidencias.


In [ ]:
!pip install -q groq

from getpass import getpass
import os

os.environ["GROQ_API_KEY"] = getpass("Ingresar API key de Groq: ")

## 2. Prompt maestro (system prompt)

Este es el prompt base que define el rol, la estructura obligatoria y las reglas de
razonamiento. Se construye a partir del contenido de los documentos POL-FIN-002 y
POL-FIN-003 (Paso 2), por lo que las instrucciones no son genéricas: reflejan exactamente los
criterios de clasificación y el formato definidos en la documentación interna simulada.


In [ ]:
SYSTEM_PROMPT = """Eres un analista financiero senior de FlexParts Manufacturing MX,
especializado en control de costos de manufactura. Tu tarea es generar reportes ejecutivos
de variación presupuestal para Gerencia, a partir de datos de presupuesto vs. gasto real y
del contexto de políticas internas que se te proporcione.

RAZONAMIENTO INTERNO (no lo muestres en la respuesta final, solo úsalo para pensar):
1. Clasifica cada variación según su magnitud: dentro de rango normal (-2.9% a +2.9%),
   moderada (+3% a +9.9%), significativa (+10% o más), ahorro saludable (-10% a -3%), o
   ahorro atípico (menor a -10%).
2. Verifica si la variación, por su magnitud o por ser sostenida 3 meses o más, activa
   alguna regla del contexto de políticas proporcionado. Si el contexto no incluye una
   política aplicable, indícalo explícitamente — nunca inventes un umbral o una regla que no
   esté en el contexto.
3. Distingue causas internas (atendibles por el responsable del centro de costo) de causas
   externas (fuera de su control), cuando el contexto lo permita.
4. Solo después de este análisis, redacta el reporte final.

ESTRUCTURA OBLIGATORIA DE LA RESPUESTA FINAL:
1. Resumen Ejecutivo (máximo 3 líneas)
2. Diagnóstico por Centro de Costo (variación en MXN y %, clasificación, causa probable)
3. Alertas de Política (solo si el contexto proporcionado activa alguna; si no, escribir
   "Sin alertas de política en el contexto disponible")
4. Recomendación (una acción concreta y accionable por cada hallazgo relevante)
5. Responsable y Siguiente Paso

REGLAS DE GROUNDING:
- Usa exclusivamente los datos numéricos y el contexto de políticas que se te proporcionen.
- Si citas una política o un umbral, debe provenir textualmente del contexto recibido.
- Si el contexto no cubre algo que sería útil mencionar, indica la limitación en vez de
  completar con supuestos.
- NUNCA inventes nombres de personas, cargos o responsables. El campo "Responsable" debe
  llenarse únicamente con el valor recibido en los DATOS de entrada, copiado tal cual. Si el
  campo Responsable no viene incluido en los DATOS, escribe exactamente "No especificado en
  los datos proporcionados" — no propongas un nombre, cargo o departamento por tu cuenta bajo
  ninguna circunstancia.

TONO: profesional, directo, sin tecnicismos innecesarios. El reporte debe ser legible para
un Gerente de Planta que no es especialista financiero. Evita juicios de valor sobre las
personas; evalúa procesos y resultados.
"""

print(SYSTEM_PROMPT)

## 3. Ejemplo few-shot

Se incluye un ejemplo completo, tomado del documento POL-FIN-003, para anclar el formato y
el nivel de detalle esperado antes de pasar el caso real.


In [ ]:
EJEMPLO_FEW_SHOT_ENTRADA = """
DATOS:
Centro de costo: Línea de Producción 2
Categoría: Mantenimiento
Presupuesto: $70,000 MXN | Real: $87,200 MXN (Septiembre)
Histórico: Julio +18.6%, Agosto +20.0%, Septiembre +24.6%
Responsable: Coordinador de Mantenimiento - J. Salinas

CONTEXTO DE POLÍTICAS RECUPERADO:
"Cuando una categoría de gasto en un mismo centro de costo presenta una variación positiva
(sobrecosto) durante 3 meses consecutivos o más, el responsable debe presentar un plan
correctivo formal a Gerencia, independientemente de si cada mes individual superó o no el
umbral de aprobación." (Política POL-FIN-001, sección 4)
"""

EJEMPLO_FEW_SHOT_SALIDA = """
1. Resumen Ejecutivo
Mantenimiento en Línea de Producción 2 muestra sobrecosto sostenido por tercer mes
consecutivo, activando la regla de variación sostenida de la Política POL-FIN-001.

2. Diagnóstico por Centro de Costo
- Línea de Producción 2 / Mantenimiento: variación de +$17,200 MXN (+24.6%) en septiembre.
  Clasificación: significativa. Tendencia sostenida desde julio (+18.6%, +20.0%, +24.6%).

3. Alertas de Política
Se activa la regla de variación sostenida (POL-FIN-001, sección 4): 3 meses consecutivos de
sobrecosto en la misma categoría y centro de costo requieren plan correctivo formal a
Gerencia, independientemente del monto individual de cada mes.

4. Recomendación
Solicitar al Coordinador de Mantenimiento un plan correctivo formal antes del cierre del
siguiente mes, desagregando el gasto entre mantenimiento correctivo y preventivo para
identificar si el sobrecosto responde a fallas puntuales o a un patrón estructural.

5. Responsable y Siguiente Paso
Responsable: Coordinador de Mantenimiento - J. Salinas.
Siguiente paso: presentar plan correctivo formal a Gerencia — fecha límite sugerida: cierre
del mes en curso.
"""

print("Ejemplo few-shot definido.")

## 4. Caso de prueba real (sin RAG conectado todavía)

Se prueba el prompt con el escenario de ahorro atípico en Línea de Producción 1 / Materia
Prima (Paso 1), pasando manualmente el fragmento de contexto correspondiente — en el Paso 5
esta recuperación será automática, vía el índice vectorial construido en el Paso 3.

**Nota sobre una corrección aplicada:** en una primera prueba, el campo `Responsable` no se
incluía en los datos de entrada. El modelo, al no recibir ese dato, inventó un nombre y cargo
que no existían en ningún lugar del contexto — una alucinación clásica de LLM. La causa raíz
resultó ser que el propio ejemplo few-shot (Sección 3) tampoco incluía el campo Responsable en
su entrada, solo en su salida, lo que enseñaba al modelo por imitación que era válido
completar ese dato por su cuenta. La corrección aplicada fue doble: (1) agregar el campo
Responsable también a la entrada del ejemplo few-shot, y (2) añadir una regla explícita de
grounding en el prompt maestro que prohíbe inventar nombres o cargos y exige responder "No
especificado en los datos proporcionados" cuando el campo no venga incluido.


In [ ]:
CASO_PRUEBA_ENTRADA = """
DATOS:
Centro de costo: Línea de Producción 1
Categoría: Materia Prima
Presupuesto: $620,000 MXN | Real: $565,100 MXN (Enero)
Histórico: variación entre -7% y -9% durante todo el año.
Responsable: Gerente de Línea 1 - R. Hernández

CONTEXTO DE POLÍTICAS RECUPERADO:
"Un ahorro sostenido y significativo (por debajo de -10%) en categorías como Mantenimiento
puede ser señal de mantenimiento diferido, lo cual representa un riesgo operativo futuro
aunque mejore el resultado financiero del mes. Todo análisis de ahorro debe evaluarse junto
con indicadores operativos, no solo financieros." (Política POL-FIN-002, sección 4)
"""

print(CASO_PRUEBA_ENTRADA)

In [ ]:
from groq import Groq

client = Groq()

respuesta = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": EJEMPLO_FEW_SHOT_ENTRADA},
        {"role": "assistant", "content": EJEMPLO_FEW_SHOT_SALIDA},
        {"role": "user", "content": CASO_PRUEBA_ENTRADA},
    ],
    temperature=0.3,
)

print(respuesta.choices[0].message.content)

## 5. Verificación del resultado

Puntos a revisar manualmente sobre la respuesta generada en la celda anterior (esta revisión
manual es la base del mecanismo de control de calidad que se automatiza en el Paso 7):

- ¿Sigue la estructura de 5 secciones obligatorias?
- ¿La variación en MXN y % coincide con los datos de entrada?
- ¿La política citada corresponde textualmente al contexto proporcionado, y no a una regla
  inventada (por ejemplo, no debería mencionar el umbral de $50,000 MXN de Mantenimiento,
  porque ese caso es de Materia Prima)?
- ¿El campo Responsable coincide exactamente con el valor recibido en los DATOS de entrada,
  sin nombres o cargos inventados?
- ¿La recomendación es una acción concreta y no solo una repetición del diagnóstico?

Una captura de pantalla de esta celda con su salida es evidencia directa para el informe de
desarrollo.


---
## Resumen técnico (Paso 4)

**Proceso realizado:** diseño de un prompt maestro con role prompting, estructura de salida
obligatoria, few-shot prompting y chain-of-thought guiado, probado de forma aislada (sin RAG
automático) usando la API gratuita de Groq (modelo Llama 3.3 70B).

**Decisiones técnicas documentadas:**
- El razonamiento paso a paso se instruye explícitamente pero se pide no mostrarlo en la
  respuesta final, para mantener el reporte ejecutivo limpio y directo.
- Las reglas de *grounding* (no inventar políticas fuera del contexto recibido, ni nombres o
  cargos de responsables) se incluyen como instrucción explícita, anticipando el requisito 7
  del caso práctico (filtrado y control de calidad, desarrollado formalmente en el Paso 7).
- Se usó Groq como proveedor de prueba por ser gratuito; el prompt es agnóstico de proveedor.

**Hallazgo real y corrección aplicada:** una primera prueba mostró que el modelo inventaba un
responsable (nombre y cargo) cuando ese dato no se incluía en la entrada. La causa raíz fue
que el propio ejemplo few-shot omitía ese campo en su entrada, enseñando por imitación un
comportamiento no deseado. Se corrigió agregando el campo Responsable a la entrada del
ejemplo few-shot y una regla explícita de grounding que prohíbe inventar nombres o cargos.
Este hallazgo, con su corrección, es evidencia directa de un ciclo real de control de calidad
sobre el prototipo.

**Evidencia generada:** la salida de la Sección 4 muestra que el modelo, dado el fragmento de
contexto sobre ahorro atípico, no confunde ni cita la política de Mantenimiento — permanece
correctamente acotado al contexto recibido, y ahora también al campo Responsable recibido.

**Siguiente paso:** Paso 5 — conectar este prompt maestro con el índice vectorial del Paso 3,
para que la recuperación de contexto sea automática en lugar de manual.
